In [2]:
import pandas as pd

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"

df = pd.read_csv(
    url,
    sep=";",
    compression="zip",
    low_memory=False
)

df.head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.0


In [4]:
import numpy as np

In [11]:
def create_sequences(df, seq_length):
    xs, ys = [], []
    
    for i in range(len(df) - seq_length):
        x = df.iloc[i:(i+seq_length), 1]
        y = df.iloc[i+seq_length, 1]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

In [12]:
create_sequences(df, 10)

(array([['17:24:00', '17:25:00', '17:26:00', ..., '17:31:00', '17:32:00',
         '17:33:00'],
        ['17:25:00', '17:26:00', '17:27:00', ..., '17:32:00', '17:33:00',
         '17:34:00'],
        ['17:26:00', '17:27:00', '17:28:00', ..., '17:33:00', '17:34:00',
         '17:35:00'],
        ...,
        ['20:50:00', '20:51:00', '20:52:00', ..., '20:57:00', '20:58:00',
         '20:59:00'],
        ['20:51:00', '20:52:00', '20:53:00', ..., '20:58:00', '20:59:00',
         '21:00:00'],
        ['20:52:00', '20:53:00', '20:54:00', ..., '20:59:00', '21:00:00',
         '21:01:00']], shape=(2075249, 10), dtype=object),
 array(['17:34:00', '17:35:00', '17:36:00', ..., '21:00:00', '21:01:00',
        '21:02:00'], shape=(2075249,), dtype='<U8'))

In [14]:
import torch
from torch.utils.data import TensorDataset

# Use create_sequences to create inputs and targets
X_train, y_train = create_sequences(df, 96)
print(X_train.shape, y_train.shape)

# Create TensorDataset
dataset_train = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32),
)
print(len(dataset_train))

(2075163, 96) (2075163,)


TypeError: can't convert np.ndarray of type numpy.object_. The only supported types are: float64, float32, float16, complex64, complex128, int64, int32, int16, int8, uint64, uint32, uint16, uint8, and bool.

In [16]:
import torch.nn as nn

In [17]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(input_size=1, 
                          hidden_size=32, 
                          num_layers=1, 
                          batch_first=True)
        self.fc = nn.Linear(32, 1)
        
    def forward(self, x):
        h0 = torch.zeros(2, x.size(0), 32)
        out = self.rnn(x, h0)
        out = self.fc(out[:, -1, :])
        return out

In [18]:
class Net(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=32,
            num_layers=2,
            batch_first=True,
        )
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        h0 = torch.zeros(2, x.size(0), 32)
        c0 = torch.zeros(2, x.size(0), 32)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

In [19]:
class Net(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.gru = nn.GRU(
        input_size=1,
        hidden_size=32,
        num_layers=2,
        batch_first=True,
        )
        self.fc = nn.Linear(32, 1)
        
    def forward(self, x):
        h0 = torch.zeros(2, x.size(0), 32)
        out, _ = self.gru(x, h0)
        out = self.fc(out[:, -1, :])
        return out